Pergunta 1:
Quantos setores censitários existem no municio do Rio de janeiro?

In [0]:
%sql
select 
count (distinct a.id_setor_censitario)
from engenharia_de_dados.gold_indicadres_consolidados as a

In [0]:
%sql
select 
count (distinct a.id_municipio)
from engenharia_de_dados.gold_indicadres_consolidados as a

Pergunta 2: 
É possível construir indicadores socioeconômicos em nível de setor censitário a partir das variáveis disponibilizadas pelo Censo Demográfico 2022?

In [0]:
%sql
SELECT
a.id_municipio,
a.densidade_populacional_municipio,
a.moradores_por_domicilio_municipio,
a.pct_indigena_municipio,
a.pct_quilombola_municipio
from engenharia_de_dados.gold_indicadres_consolidados as a
limit 10

Pergunta 3:
É possível mapear regiões potenciais a partir de uma clusterização k-means dos setores?

In [0]:
%sql
select * from engenharia_de_dados.gold_indicadres_consolidados limit 5

In [0]:
%sql
SELECT
    column_name,
    data_type
FROM information_schema.columns
WHERE table_schema = 'engenharia_de_dados'
  AND table_name = 'gold_indicadres_consolidados'
ORDER BY ordinal_position

transformando em data frame para conseguir usar k-means via python


In [0]:
df = spark.table(
    "engenharia_de_dados.gold_indicadres_consolidados"
)

In [0]:
df.printSchema()

aplicação do algoritmo

In [0]:
id_cols = [
    "id_municipio",
    "sigla_uf"
]

features = [
    "densidade_populacional_municipio",
    "moradores_por_domicilio_municipio",
    "pct_indigena_municipio",
    "pct_quilombola_municipio",
    "razao_sexo_municipio",
    "taxa_alfabetizacao_municipio",
    "idade_mediana_municipio",
    "indice_envelhecimento_municipio"
]

In [0]:
df_modelo = df.select(
    *id_cols,
    *features
)

In [0]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=features,
    outputCol="features_raw"
)

df_assembled = assembler.transform(df_modelo)

In [0]:
from pyspark.ml.feature import StandardScaler

scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withMean=True,
    withStd=True
)

scaler_model = scaler.fit(df_assembled)

df_scaled = scaler_model.transform(df_assembled)

In [0]:
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

evaluator = ClusteringEvaluator(
    featuresCol="features",
    predictionCol="prediction",
    metricName="silhouette",
    distanceMeasure="squaredEuclidean"
)

resultados = []

for k in range(2, 11):

    kmeans = KMeans(
        k=k,
        seed=42,
        featuresCol="features",
        predictionCol="prediction"
    )

    model = kmeans.fit(df_scaled)

    predictions = model.transform(df_scaled)

    silhouette = evaluator.evaluate(predictions)

    resultados.append((k, silhouette))

resultados

O Silhouette Score mede, de forma resumida, quão bem cada observação está dentro do próprio cluster em comparação com os clusters vizinhos.

A escala vai de -1 a 1:

próximo de 1 → clusters bem separados e observações bem alocadas

próximo de 0 → clusters se sobrepõem bastante

abaixo de 0 → pode haver observações mal agrupadas

> No meu resultado, a melhor distribuição ficou em 7 com 0.64, então usarei k = 7.

In [0]:
kmeans_final = KMeans(
    k=7,
    seed=42,
    featuresCol="features",
    predictionCol="cluster"
)

modelo_final = kmeans_final.fit(df_scaled)

In [0]:
df_clusters = modelo_final.transform(df_scaled)

In [0]:
display(
    df_clusters
    .groupBy("cluster")
    .count()
    .orderBy("cluster")
)

In [0]:
from pyspark.sql.functions import avg, col

display(
    df_clusters
    .groupBy("cluster")
    .agg(
        *[
            avg(col(c)).alias(c)
            for c in features
        ]
    )
    .orderBy("cluster")
)